<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Operaciones de Aprendizaje Automático III**
# **Clase 4, QLoRA y adaptadores**

**Contexto:**
Un equipo necesita tres cosas sobre las reseñas que dejan sus clientes:
* clasificarlas
* extraer de cada una una ficha estructurada para el sistema de reclamos
* redactar la respuesta que se le manda al cliente

No entrenaremos 3 modelos, solo uno y usaremos 3 adaptadores encima. En
producción se carga el modelo una vez y se conmuta el adaptador según lo que haga falta.


Los tres adaptadores reciben el mismo prompt: el mismo mensaje de sistema y el mismo texto de la reseña, la única variable es el adaptador, y eso significa que el adaptador es el comportamiento.


### **Instalamos lo necesario**

Instalamos versiones fijas de transformers y peft


In [ ]:
!pip install -q -U "transformers>=4.45" "peft>=0.13" datasets accelerate bitsandbytes

import torch, transformers, peft

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")


torch: 2.11.0+cu128
transformers: 5.17.0
peft: 0.21.0
GPU: Tesla T4


### **Cargamos el modelo base en 4 bits**



In [ ]:
import gc, json, os, time, torch
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorForLanguageModeling)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

In [ ]:
MODELO = "Qwen/Qwen2.5-0.5B-Instruct"

DTYPE = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float16"
USA_BF16, USA_FP16 = DTYPE == "bfloat16", DTYPE == "float16"
EN_GPU = torch.cuda.is_available()

print("precision de computo:", DTYPE)

precision de computo: bfloat16


In [ ]:
BNB = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=getattr(torch, DTYPE),
)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODELO)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

In [ ]:
def cargar_base():
    #Modelo base en 4 bits, congelado, ninguno de sus pesos se entrena
    return AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=BNB, device_map="auto")

In [ ]:
base = cargar_base()
MB_MODELO = base.get_memory_footprint() / 1e6

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

### **Entrada**

* El prompt es siempre este, un mensaje de sistema neutro y la reseña
* No vamos a pedir una tarea
* La tarea la va a poner el adaptador
* La generación es determinista (do_sample=False), así las comparaciones son limpias

In [ ]:
sistema = "Procesas reseñas de clientes de una tienda online."
review = "El monitor se apaga solo cada veinte minutos. Es inusable, quiero devolverlo."

In [ ]:
def prompt_de(resena):
    mensajes = [{"role": "system", "content": sistema},
                {"role": "user",   "content": review}]
    return tok.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)

def responder(modelo, resena, max_tokens=70):
    entrada = tok(prompt_de(resena), return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, pad_token_id=tok.pad_token_id)

    return tok.decode(salida[0][entrada.input_ids.shape[1]:], skip_special_tokens=True).strip()

def mostrar(etiqueta, texto):
    print(etiqueta)
    print(texto)

In [ ]:
print("Reseña:", review)
mostrar("Modelo base, sin adaptador:", responder(base, review))

Reseña: El monitor se apaga solo cada veinte minutos. Es inusable, quiero devolverlo.
Modelo base, sin adaptador:
Lo siento que no puedo ayudarte con eso.


Lo que sale es una respuesta conversacional y completamente inservible para un
sistema que espera una etiqueta o un JSON, este es el problema que resuelven los adaptadores.


### **Los datos**

Las mismas reseñas se usan para los tres adaptadores, lo único que cambia es la
respuesta esperada:

| adaptador | salida | para qué |
|---|---|---|
| sentimiento | una palabra: POSITIVO, NEUTRO, NEGATIVO | tablero de métricas |
| ficha | un JSON con producto, problema y urgencia | sistema de reclamos |
| respuesta | una o dos oraciones para el cliente | bandeja de atención |

Esto no es estilo, son tres tareas con salidas de naturaleza diferente: una etiqueta,
una estructura y un texto.


In [ ]:
resenas = [
    "Llegó tres días tarde y la caja venía abierta. El producto igual funciona.",
    "Los audífonos son excelentes, la batería dura todo el día.",
    "La licuadora dejó de andar a la semana. Quiero la devolución.",
    "Cumple lo que promete, nada más que agregar.",
    "El monitor tiene un pixel muerto en la esquina. Molesta poco pero está.",
    "Me encantó la silla, muy cómoda para trabajar muchas horas.",
    "Pedí una bicicleta azul y llegó roja. Necesito el cambio esta semana.",
    "Buen precio y calidad aceptable. La volvería a comprar.",
    "La lámpara llegó bien pero el cable es más corto de lo que decía.",
    "Tres semanas esperando y nadie responde los mensajes. Un desastre.",
    "El café es rico pero el paquete venía mal sellado.",
    "Envío rapidísimo, en dos días lo tenía en casa.",
    "El rompecabezas venía incompleto, faltan cuatro piezas.",
    "La chamarra abriga bien y el talle es el correcto.",
    "Funciona, aunque el manual está solo en inglés.",
    "El monitor se apaga solo cada veinte minutos. Es inusable.",
]

etiquetas = [
    "NEGATIVO",
    "POSITIVO",
    "NEGATIVO",
    "NEUTRO",
    "NEGATIVO",
    "POSITIVO",
    "NEGATIVO",
    "POSITIVO",
    "NEUTRO",
    "NEGATIVO",
    "NEUTRO",
    "POSITIVO",
    "NEGATIVO",
    "POSITIVO",
    "NEUTRO",
    "NEGATIVO",
]

fichas = [
    {"producto": "no indicado", "problema": "demora y embalaje abierto", "urgencia": "media"},
    {"producto": "audifonos",   "problema": "ninguno",                   "urgencia": "baja"},
    {"producto": "licuadora",   "problema": "deja de funcionar",         "urgencia": "alta"},
    {"producto": "no indicado", "problema": "ninguno",                   "urgencia": "baja"},
    {"producto": "monitor",     "problema": "pixel muerto",              "urgencia": "baja"},
    {"producto": "silla",       "problema": "ninguno",                   "urgencia": "baja"},
    {"producto": "bicicleta",   "problema": "color equivocado",          "urgencia": "alta"},
    {"producto": "no indicado", "problema": "ninguno",                   "urgencia": "baja"},
    {"producto": "lampara",     "problema": "cable mas corto",           "urgencia": "baja"},
    {"producto": "no indicado", "problema": "demora y sin respuesta",    "urgencia": "alta"},
    {"producto": "cafe",        "problema": "paquete mal sellado",       "urgencia": "media"},
    {"producto": "no indicado", "problema": "ninguno",                   "urgencia": "baja"},
    {"producto": "rompecabezas","problema": "piezas faltantes",          "urgencia": "media"},
    {"producto": "chamarra",    "problema": "ninguno",                   "urgencia": "baja"},
    {"producto": "no indicado", "problema": "manual solo en ingles",     "urgencia": "baja"},
    {"producto": "monitor",     "problema": "se apaga solo",             "urgencia": "alta"},
]

respuestas = [
    "Lamentamos la demora y el estado del paquete. Te enviamos un embalaje nuevo sin costo.",
    "Gracias por tu comentario. Nos alegra que la batería te resulte suficiente.",
    "Lamentamos la falla. Gestionamos la devolución y te contactamos en 24 horas.",
    "Gracias por tu reseña. Quedamos a disposición si necesitas algo más.",
    "Lamentamos el pixel muerto. Si preferís el cambio, lo gestionamos sin costo.",
    "Gracias por contarnos. Nos alegra que la silla te acompañe en la jornada.",
    "Lamentamos el error en el color. Ya iniciamos el cambio para esta semana.",
    "Gracias por tu compra y por volver. Te esperamos.",
    "Gracias por el aviso. Corregimos la descripción del cable en la ficha.",
    "Lamentamos la falta de respuesta. Derivamos tu caso con prioridad alta.",
    "Gracias por avisar del sellado. Te enviamos un paquete de reemplazo.",
    "Gracias por tu comentario. Seguimos trabajando para mantener los tiempos.",
    "Lamentamos las piezas faltantes. Te enviamos el repuesto esta semana.",
    "Gracias por tu reseña. Nos alegra que el talle sea el correcto.",
    "Gracias por el aviso. Estamos preparando el manual en español.",
    "Lamentamos la falla del monitor. Gestionamos el reemplazo de inmediato.",
]

respuestas_ficha = [json.dumps(f, ensure_ascii=False) for f in fichas]

def armar(resenas, salidas):
    #cada ejemplo es la conversación completa: mismo prompt, salida distinta
    filas = []
    for r, s in zip(resenas, salidas):
        mensajes = [{"role": "system", "content": sistema},
                    {"role": "user",   "content": r},
                    {"role": "assistant", "content": s}]
        filas.append({"text": tok.apply_chat_template(mensajes, tokenize=False)})
    return Dataset.from_list(filas)

TAREAS = {
    "sentimiento": armar(resenas, etiquetas),
    "ficha": armar(resenas, respuestas_ficha),
    "respuesta": armar(resenas, respuestas),
}

print("adaptadores a entrenar:", list(TAREAS))
print("ejemplos en cada uno  :", len(resenas))
print("la misma reseña, las tres salidas esperadas")
print("reseña:", resenas[2])
print("sentimiento:", etiquetas[2])
print("ficha:", respuestas_ficha[2])
print("respuesta:", respuestas[2])


adaptadores a entrenar: ['sentimiento', 'ficha', 'respuesta']
ejemplos en cada uno  : 16
la misma reseña, las tres salidas esperadas
reseña: La licuadora dejó de andar a la semana. Quiero la devolución.
sentimiento: NEGATIVO
ficha: {"producto": "licuadora", "problema": "deja de funcionar", "urgencia": "alta"}
respuesta: Lamentamos la falla. Gestionamos la devolución y te contactamos en 24 horas.


### **Entrenamos QLoRA**

Un adaptador LoRA agrega, junto a cada matriz de peso congelada, dos matrices chicas cuyo producto tiene la misma forma que la originaL, solo esas matrices se entrenan: con rango 16 sobre siete proyecciones, alrededor del 2 % de los parámetros.

La misma función se llama tres veces, con los mismos hiperparámetros y datos distintos.

El modelo base es el mismo y no se toca en ningun momento

In [ ]:
LORA = dict(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj",
                    "k_proj",
                    "v_proj",
                    "o_proj",
                    "gate_proj",
                    "up_proj",
                    "down_proj"],

    task_type="CAUSAL_LM", bias="none")

In [ ]:
def entrenar_adaptador(dataset, salida, epocas=10, lr=3e-4):
    modelo = get_peft_model(prepare_model_for_kbit_training(cargar_base()),
                            LoraConfig(**LORA))
    modelo.config.use_cache = False
    print(f"--- entrenando {salida} ---")
    modelo.print_trainable_parameters()

    ds = dataset.map(lambda b: tok(b["text"], truncation=True, max_length=256),
                     batched=True, remove_columns=["text"])

    args = TrainingArguments(
        output_dir=f"{salida}-tmp",
        num_train_epochs=epocas,
        per_device_train_batch_size=2,
        learning_rate=lr,
        fp16=USA_FP16,
        bf16=USA_BF16,
        logging_steps=40,
        save_strategy="no",
        report_to=[],
        lr_scheduler_type="constant")

    Trainer(model=modelo,
            args=args,
            train_dataset=ds,
            data_collator=DataCollatorForLanguageModeling(tok, mlm=False)).train()

    modelo.save_pretrained(salida)
    del modelo
    gc.collect()
    torch.cuda.empty_cache()
    print(f"guardado en {salida}")

for nombre, ds in TAREAS.items():
    entrenar_adaptador(ds, f"adaptadores/{nombre}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

--- entrenando adaptadores/sentimiento ---
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


Map:   0%|          | 0/16 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
40,0.845726
80,0.107302


...

--- entrenando adaptadores/respuesta ---
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


Map:   0%|          | 0/16 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
40,0.876626
80,0.093937


guardado en adaptadores/respuesta


### **cuanto pesa lo que acabamos de entrenar?**

In [ ]:
def mb(ruta):
    return sum(os.path.getsize(os.path.join(d, f))
               for d, _, fs in os.walk(ruta) for f in fs) / 1e6

print(f"modelo base en 4 bits: {MB_MODELO:8.1f} MB en memoria")
suma = 0

for nombre in TAREAS:
    peso = mb(f"adaptadores/{nombre}")
    suma += peso
    print(f"adaptador {nombre}: {peso:8.1f} MB en disco")

print(f"{'los tres juntos'}: {suma:8.1f} MB")
print()
print(f"Tres modelos separados costarian unos {MB_MODELO * 3:.0f} MB.")
print(f"Un modelo más tres adaptadores cuestan {MB_MODELO + suma:.0f} MB.")

modelo base en 4 bits:    451.3 MB en memoria
adaptador sentimiento:     35.2 MB en disco
adaptador ficha:     35.2 MB en disco
adaptador respuesta:     35.2 MB en disco
los tres juntos:    105.7 MB

Tres modelos separados costarian unos 1354 MB.
Un modelo más tres adaptadores cuestan 557 MB.


### **los tres montados a la vez sobre una sola copia del modelo**

Se carga el modelo base una sola vez y se le montan los tres adaptadores, cada uno con su nombre.
* **PeftModel.from_pretrained** monta el primero
* **load_adapter** agrega los que siguen
* a partir de ahí **set_adapter** elige cuál está activo.

El costo de tener varios adaptadores cargados es despreciable frente al modelo


In [ ]:
del base
gc.collect()
torch.cuda.empty_cache()

def memoria_mb():
    return torch.cuda.memory_allocated() / 1e6 if EN_GPU else float("nan")

base_solo = cargar_base()
antes = memoria_mb()
print(f"memoria con el modelo solo: {antes:8.1f} MB")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

memoria con el modelo solo:    477.0 MB


In [ ]:
nombres = list(TAREAS)
modelo = PeftModel.from_pretrained(base_solo,
                                   f"adaptadores/{nombres[0]}",
                                   adapter_name=nombres[0])

for n in nombres[1:]:
    modelo.load_adapter(f"adaptadores/{n}", adapter_name=n)

modelo.config.use_cache = True

despues = memoria_mb()

print(f"memoria con los tres adaptadores: {despues:8.1f} MB")
print(f"costo de los tres adaptadores: {despues - antes:8.1f} MB\n")
print("adaptadores montados:", list(modelo.peft_config.keys()))
print("activo ahora:", modelo.active_adapter)

memoria con los tres adaptadores:    582.6 MB
costo de los tres adaptadores:    105.6 MB

adaptadores montados: ['sentimiento', 'ficha', 'respuesta']
activo ahora: sentimiento


### **tres salidas**

Mismo prompt en los tres casos, lo único que cambia entre una línea y la siguiente es set_adapter


In [ ]:
print("Reseña:", review, "\n")

TOPES = {"sentimiento": 12,
         "ficha": 70,
         "respuesta": 45}

for nombre in modelo.peft_config:
    modelo.set_adapter(nombre)
    mostrar(f"Adaptador, {nombre}", responder(modelo, review, TOPES[nombre]))

with modelo.disable_adapter():
    mostrar("Sin adaptador, el modelo base original", responder(modelo, review))


Reseña: El monitor se apaga solo cada veinte minutos. Es inusable, quiero devolverlo. 

Adaptador, sentimiento
NEGATIVO
Adaptador, ficha
{"producto": "monitor", "problema": "se apaga solo", "urgencia": "alta"}
Adaptador, respuesta
Lamentamos la falla del monitor. Gestionamos el reemplazo de inmediato sin costo.
Sin adaptador, el modelo base original
Lo siento que no puedo ayudarte con eso.


### **un servicio mínimo que atiende las tres tareas**

Así se vería el bucle de un servicio real:

* llegan pedidos mezclados
* cada uno pide una tarea distinta
* el servicio conmuta el adaptador antes de responder

una sola copia del modelo en memoria con tres comportamientos, las reseñas de esta celda son nuevas, o sea el modelo no las vio al entrenar


In [ ]:
cola = [
    ("ficha", "La pelota vino desinflada y sin la válvula."),
    ("sentimiento", "Excelente atención, resolvieron el cambio en un día."),
    ("respuesta", "El atlas está bien impreso pero llegó con la tapa doblada."),
    ("sentimiento", "Hace dos meses que espero la factura y nadie contesta."),
    ("ficha", "Hace dos meses que espero la factura y nadie contesta."),
    ("respuesta", "El teclado responde perfecto, muy buena compra."),
    ("ficha", "El producto está bien, el envío fue caro."),
]

def atender(tarea, resena):
    # elige y responde
    modelo.set_adapter(tarea)
    return responder(modelo, resena, TOPES[tarea])

validos = total_fichas = 0
for tarea, resena in cola:
    salida = atender(tarea, resena)
    marca = ""
    if tarea == "ficha":
        total_fichas += 1
        try:
            json.loads(salida)
            marca, validos = "  [json ok]", validos + 1
        except Exception:
            marca = "  [json roto]"
    print(f"[{tarea}]{marca}")
    print("reseña:", resena)
    print("salida:", salida.replace("\n", " ")[:110])
    print("_"*50)

print(f"fichas con JSON válido: {validos} de {total_fichas}")

[ficha]  [json ok]
reseña: La pelota vino desinflada y sin la válvula.
salida: {"producto": "monitor", "problema": "se apaga solo", "urgencia": "alta"}
__________________________________________________
[sentimiento]
reseña: Excelente atención, resolvieron el cambio en un día.
salida: NEGATIVO
__________________________________________________
[respuesta]
reseña: El atlas está bien impreso pero llegó con la tapa doblada.
salida: Lamentamos la falla del monitor. Gestionamos el reemplazo de inmediato sin costo.
__________________________________________________
[sentimiento]
reseña: Hace dos meses que espero la factura y nadie contesta.
salida: NEGATIVO
__________________________________________________
[ficha]  [json ok]
reseña: Hace dos meses que espero la factura y nadie contesta.
salida: {"producto": "monitor", "problema": "se apaga solo", "urgencia": "alta"}
__________________________________________________
[respuesta]
reseña: El teclado responde perfecto, muy buena compra.
salida:

**conclusiones**
- El modelo base se cuantiza una vez, queda congelado y nunca se entrena
- Lo que se entrena son adaptadores
- Tres adaptadores conviven sobre una sola copia del modelo
- Cambiar entre ellos toma milisegundos
- El adaptador es el comportamiento: con el prompt idéntico, la salida cambia de naturaleza.
- Apagar el adaptador devuelve el modelo original, sin una segunda copia

**lo que nos falta...**

- el objetivo de entrenamiento que se usó acá es imitar ejemplos
- funciona porque alguien pudo escribir la etiqueta, la ficha y la respuesta correctas de cada reseña

---
* **muchas veces nadie puede escribir la respuesta correcta, no hay una sola, o escribirla cuesta más que el problema**
* **... pero casi siempre alguien puede mirar dos respuestas y decir cuál prefiere**
* **para entrenar con eso hace falta otro objetivo, y es el tema del DPO**